# Comparative Sentiment Analysis: Indonesian E-Commerce Reviews (Scopus Pipeline)
## Notebook 02: Preprocessing, Normalization, Labeling & Stratified Splitting

```text
Author: Muhammad Farrel Haidar
Project: AI PR & KOL Specialist DSS
Date: 2026-10-14
Study: Comparative Sentiment Analysis (IndoBERT vs Tree-Based Models)
Target: Preprocessed Corpus of 12,000+ Clean Indonesian E-Commerce Reviews
```

---
### 1. Metodologi Preprocessing & Standar Publikasi Scopus
Pada tahap ini, korpus ulasan mentah dari Google Play Store (Tokopedia, Shopee, Lazada, Bukalapak) diproses melalui serangkaian tahapan rekayasa data (*data cleaning pipeline*):
1. **Deduplikasi**: Menghilangkan ulasan redundan berdasarkan kesamaan konten teks ulasan.
2. **Panjang Minimum**: Mengeliminasi ulasan dengan panjang karakter < 10 (misal: spam titik, angka acak, satu kata kosong).
3. **Language Identification**: Memastikan hanya ulasan berbahasa Indonesia yang dipertahankan menggunakan leksikon dan `langdetect`.
4. **Normalisasi Slang & Teks**: Lowercase, pembersihan URL/Mention/Hashtag/Emoji, reduksi karakter berulang (e.g., 'baguuus' -> 'bagus'), dan mapping kamus slang bahasa Indonesia.
5. **Labeling Konsisten**: Pemetaan rating bintang 1-2 menjadi *Negatif (0)*, 3 menjadi *Netral (1)*, dan 4-5 menjadi *Positif (2)*.
6. **Stratified Split (80/10/10)**: Pemisahan data latih, validasi, dan uji dengan mempertahankan proporsi kelas sentimen menggunakan `seed=42`.

In [ ]:
# Author: Muhammad Farrel Haidar
# Project: AI PR & KOL Specialist DSS
# Date: 2026-10-14

import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from langdetect import detect, DetectorFactory

# Set reproducibility seed
DetectorFactory.seed = 42
np.random.seed(42)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
print("Environment initialized successfully.")

### 2. Kamus Slang Bahasa Indonesia & Modul Pembersihan Teks

In [ ]:
SLANG_DICT = {
    "gak": "tidak", "ga": "tidak", "nggak": "tidak", "ngga": "tidak", "gk": "tidak", "tdk": "tidak",
    "g": "tidak", "nda": "tidak", "ndak": "tidak", "kagak": "tidak", "kaga": "tidak",
    "bgt": "banget", "bgtt": "banget", "bgtz": "banget", "bnget": "banget",
    "udh": "sudah", "udah": "sudah", "sdh": "sudah", "dah": "sudah",
    "bs": "bisa", "bisaa": "bisa", "bsa": "bisa",
    "bener": "benar", "bnr": "benar", "beneran": "benar",
    "krn": "karena", "karna": "karena", "coz": "karena", "krena": "karena",
    "tp": "tapi", "tpi": "tapi", "tapii": "tapi",
    "sy": "saya", "gua": "saya", "gw": "saya", "aku": "saya", "gue": "saya", "gwe": "saya",
    "km": "kamu", "lu": "kamu", "loe": "kamu", "ente": "kamu",
    "bgs": "bagus", "bgus": "bagus", "bagusss": "bagus", "mantab": "mantap", "mantul": "mantap",
    "top": "mantap", "keren": "bagus", "kece": "bagus", "ok": "oke", "okeey": "oke",
    "kecewaa": "kecewa", "kcwa": "kecewa", "jelek": "buruk", "jlk": "buruk", "parah": "buruk", "hancur": "buruk",
    "batal": "batal", "cancel": "batal",
    "slow": "lambat", "lemot": "lambat", "lola": "lambat", "lelet": "lambat",
    "makasih": "terima kasih", "tq": "terima kasih", "thx": "terima kasih", "thanks": "terima kasih", "tengkyu": "terima kasih",
    "ongkir": "ongkos kirim", "freeongkir": "gratis ongkos kirim", "gratisongkir": "gratis ongkos kirim",
    "seller": "penjual", "buyer": "pembeli", "cs": "layanan pelanggan",
    "respon": "respons", "bales": "balas", "bls": "balas", "dibalas": "direspons",
    "dgn": "dengan", "dg": "dengan", "utk": "untuk", "yg": "yang", "yng": "yang",
    "aja": "saja", "ajah": "saja", "aj": "saja", "doang": "saja", "tok": "saja",
    "emg": "memang", "emang": "memang", "pdhl": "padahal", "sm": "sama", "sma": "sama",
    "blm": "belum", "blom": "belum", "belom": "belum",
    "bikin": "buat", "brg": "barang", "brang": "barang", "dtg": "datang", "ampe": "sampai",
    "nyampe": "sampai", "sampe": "sampai", "smpe": "sampai", "expedisi": "ekspedisi",
    "kurir": "kurir", "rusak": "rusak", "pecah": "rusak", "ilang": "hilang",
    "apk": "aplikasi", "apikasi": "aplikasi", "app": "aplikasi", "apps": "aplikasi",
    "diskon": "diskon", "promo": "promosi", "voucher": "kupon", "murah": "terjangkau",
    "mahal": "mahal", "kemahalan": "terlalu mahal", "rugi": "rugi", "penipuan": "penipuan",
    "tipu": "tipu", "bohong": "bohong", "nipuu": "menipu"
}

INDONESIAN_MARKERS = set([
    "yang", "dan", "di", "ini", "ke", "dari", "bisa", "bagus", "banget", "tidak",
    "kecewa", "paket", "barang", "ongkir", "aplikasi", "beli", "pesan", "toko",
    "kurir", "pengiriman", "harga", "murah", "mantap", "jelek", "rusak", "puas",
    "lama", "cepat", "admin", "update", "promo", "gratis", "suka", "tolong",
    "gagal", "bintang", "saya", "aku", "kami", "kamu", "gak", "udah", "bgt",
    "nya", "kalo", "kalau", "tapi", "tp", "sy", "ada", "itu", "lagi", "sudah",
    "buat", "mau", "sama", "kok", "padahal", "belum", "masih", "selalu"
])

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"(.)\1{2,}", r"\1", text)
    words = text.split()
    normalized = [SLANG_DICT.get(w, w) for w in words]
    text = " ".join(normalized)
    return re.sub(r"\s+", " ", text).strip()

def is_indonesian(text: str) -> bool:
    if not isinstance(text, str) or len(text.strip()) < 10:
        return False
    tokens = set(re.findall(r"\b[a-z]+\b", text.lower()))
    if len(tokens.intersection(INDONESIAN_MARKERS)) >= 1:
        return True
    try:
        return detect(text) in ["id", "jw", "su"]
    except Exception:
        return False

### 3. Load Dataset Mentah & Filtering Multi-Tahap

In [ ]:
raw_dir = "../data/raw" if os.path.exists("../data/raw") else "data/raw"
files = glob.glob(os.path.join(raw_dir, "*_reviews.csv"))
print(f"File mentah terdeteksi: {len(files)}")

dfs = [pd.read_csv(f) for f in files]
raw_df = pd.concat(dfs, ignore_index=True)
initial_total = len(raw_df)
print(f"Total Baris Mentah: {initial_total:,}")

# 1. Drop duplikat konten teks
df_dedup = raw_df.drop_duplicates(subset=["content"]).copy()
print(f"Setelah Deduplikasi: {len(df_dedup):,} (Dihapus: {initial_total - len(df_dedup):,})")

# 2. Drop ulasan < 10 karakter
df_len = df_dedup[df_dedup["content"].fillna("").astype(str).str.len() >= 10].copy()
print(f"Setelah Filter Karakter >= 10: {len(df_len):,} (Dihapus: {len(df_dedup) - len(df_len):,})")

# 3. Language Filtering
df_len["is_id"] = df_len["content"].apply(is_indonesian)
df_clean_lang = df_len[df_len["is_id"]].copy()
print(f"Setelah Language Filter: {len(df_clean_lang):,} (Dihapus: {len(df_len) - len(df_clean_lang):,})")

# 4. Normalisasi Teks
df_clean_lang["cleaned_text"] = df_clean_lang["content"].apply(clean_text)
df_final = df_clean_lang[df_clean_lang["cleaned_text"].str.len() >= 5].copy()

# 5. Labeling: 1-2=Negatif (0), 3=Netral (1), 4-5=Positif (2)
def get_label_tuple(s):
    if s in [1, 2]: return 0, "negatif"
    elif s == 3: return 1, "netral"
    else: return 2, "positif"

df_final["label"] = df_final["score"].apply(lambda s: get_label_tuple(s)[0])
df_final["sentiment"] = df_final["score"].apply(lambda s: get_label_tuple(s)[1])
print(f"\nFinal Cleaned Corpus: {len(df_final):,} baris (Retensi: {len(df_final)/initial_total*100:.2f}%)")

### 4. Stratified Train / Validation / Test Splitting (80 / 10 / 10)

In [ ]:
cols = ["reviewId", "userName", "content", "cleaned_text", "score", "label", "sentiment", "thumbsUpCount", "at", "appVersion", "app_name"]

# 80% train, 20% temp
train_df, temp_df = train_test_split(df_final, test_size=0.20, random_state=42, stratify=df_final["label"])
# 10% val, 10% test
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["label"])

splits_dir = "../data/splits" if os.path.exists("../data/splits") else "data/splits"
os.makedirs(splits_dir, exist_ok=True)

train_df[cols].to_csv(os.path.join(splits_dir, "train.csv"), index=False)
val_df[cols].to_csv(os.path.join(splits_dir, "val.csv"), index=False)
test_df[cols].to_csv(os.path.join(splits_dir, "test.csv"), index=False)

print(f"Train size : {len(train_df):,} ({len(train_df)/len(df_final)*100:.1f}%)")
print(f"Val size   : {len(val_df):,} ({len(val_df)/len(df_final)*100:.1f}%)")
print(f"Test size  : {len(test_df):,} ({len(test_df)/len(df_final)*100:.1f}%)")

# Sample Validasi Manual (10%)
manual_sample = df_final.sample(frac=0.10, random_state=42).copy()
manual_sample["label_manual"] = ""
sample_cols = ["reviewId", "app_name", "score", "label", "sentiment", "cleaned_text", "label_manual"]
manual_sample[sample_cols].to_csv("../data/manual_validation_sample.csv" if os.path.exists("../data") else "data/manual_validation_sample.csv", index=False)
print(f"Manual validation sample exported: {len(manual_sample):,} rows")

### 5. Table 1 (Dataset Statistics) & Figure 1 (Visualisasi Distribusi)

In [ ]:
# Tampilkan Tabel Statistik Korpus
t1_path = "../paper_assets/table1_dataset_statistics.csv" if os.path.exists("../paper_assets") else "paper_assets/table1_dataset_statistics.csv"
if os.path.exists(t1_path):
    t1 = pd.read_csv(t1_path)
    display(t1)

# Visualisasi Distribusi Figure 1
plt.figure(figsize=(10, 6), dpi=300)
palette = {"negatif": "#d9534f", "netral": "#f0ad4e", "positif": "#5cb85c"}
ax = sns.countplot(
    data=df_final,
    x="app_name",
    hue="sentiment",
    palette=palette,
    order=["tokopedia", "shopee", "lazada", "bukalapak"]
)
plt.title("Figure 1: Distribution of Sentiment Labels Across E-Commerce Applications", fontsize=14, pad=15, fontweight="bold")
plt.xlabel("E-Commerce Application", fontsize=12, labelpad=10)
plt.ylabel("Number of Reviews", fontsize=12, labelpad=10)
plt.legend(title="Sentiment Class", title_fontsize="11", loc="upper right")
plt.xticks(ticks=[0, 1, 2, 3], labels=["Tokopedia", "Shopee", "Lazada", "Bukalapak"], fontsize=11)
plt.tight_layout()
plt.show()